
# Analisis Data Bersih — Skrining Bantul 2025

Notebook ini menggabungkan tiga analisis atas data yang sudah dibersihkan oleh
`Persiapan_Data_Revised.ipynb` menjadi satu dokumen.

| Bagian | Isi |
|---|---|
| **I** | **Profil Data Bersih** — dampak pembersihan, demografi, antropometri, prevalensi per subkelompok, komorbiditas |
| **II** | **Klasifikasi BMI** — standar WHO & CDC, hasil klasifikasi, WHO-8 vs WHO-4 |
| **III** | **Hubungan Antar Variabel** — kekuatan hubungan tiap variabel bebas dengan tiap penyakit |
| **Kesimpulan** | temuan penutup ketiga bagian, dikumpulkan jadi satu |
| **Lampiran A** | metode, ukuran per pasangan tipe, dan cara membaca grafik risiko |

Urutan ini disengaja: Bagian I memotret dulu populasinya, Bagian II menjelaskan
standar di balik kategori status gizi yang dipakai di sana, dan Bagian III mengukur
kekuatan hubungan yang muncul di kedua bagian sebelumnya.

Di luar notebook ini: data sebelum pembersihan dibahas di `Analisis Data Mentah.ipynb`,
dan pemodelan di `Script penyakitan/Stacked Revisi.ipynb`.

---

### Tiga catatan yang berlaku untuk SELURUH angka di notebook ini

1. **Semua angka berlaku per catatan skrining, bukan per orang.** Kolom `id` sengaja
   diacak untuk melindungi data asli BPJS, sehingga deduplikasi tingkat individu tidak
   mungkin dilakukan. Bila ada orang yang tercatat lebih dari sekali, ia terhitung lebih
   dari sekali.

2. **Setiap target punya rentang usia sendiri.** `Tidak Dapat Pertanyaan` berarti
   responden tidak ditanyai karena berada di luar sasaran usia instrumen, bukan berarti
   datanya hilang. Diabetes & Hipertensi hanya mencakup usia 15+, Kanker Paru 45+,
   sedangkan TBC mencakup semua usia. Rentang ini dicantumkan pada setiap grafik.

3. **Prevalensi "berisiko diabetes" bukan temuan epidemiologis yang berdiri sendiri.**
   Label tersebut praktis merupakan aturan `IMT > 25` (lihat **Bagian II.5**), sehingga sebagian besar hanyalah pernyataan ulang dari prevalensi overweight.

In [ ]:
# ===================== IMPORTS & LOAD =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Bantul"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import bmi_standards as bs      # standar BMI & batas plausibilitas per usia
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 190)

DATA_DIR = cfg.folder_data(WILAYAH)

# df_raw hanya dipakai Bagian I.1-I.2 untuk mengukur dampak pembersihan.
df_raw = cfg.baca_mentah(WILAYAH)
df = pd.read_csv(os.path.join(DATA_DIR, "dataset_master.csv"))

TARGETS = [
    "Berisiko Diabetes Melitus",
    "Berisiko Hipertensi",
    "Berisiko Kanker Paru",
    "Berisiko TBC",
]

# Rentang usia efektif tiap target (hasil gerbang eligibilitas kuesioner).
ELIGIBLE = {
    "Berisiko Diabetes Melitus": "15+",
    "Berisiko Hipertensi": "15+",
    "Berisiko Kanker Paru": "45+",
    "Berisiko TBC": "semua usia",
}

# Bentuk yang sama, disusun untuk Bagian III: nama pendek -> (kolom, rentang usia).
# Namanya dibedakan dari TARGETS karena tipenya berbeda (dict vs list) dan
# keduanya dipakai di dokumen yang sama.
TARGET_INFO = {
    "Diabetes": ("Berisiko Diabetes Melitus", "15+"),
    "Hipertensi": ("Berisiko Hipertensi", "15+"),
    "Kanker Paru": ("Berisiko Kanker Paru", "45+"),
    "TBC": ("Berisiko TBC", "semua usia"),
}

# Variabel bebas yang dianalisis di Bagian III.
NUM = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
CAT = ["Jeniskelamin", "segmen"]

print(f"Sumber standar : {bs.SOURCE_URL}")
print(f"Batas usia anak: {bs.CHILD_MIN_AGE} - {bs.CHILD_MAX_AGE} tahun")
print(f"Data mentah    : {df_raw.shape[0]:,} baris")
print(f"Data bersih    : {df.shape[0]:,} catatan skrining")
display(df.head())



# Bagian I — Profil Data Bersih

Bagian ini menganalisis data yang sudah dibersihkan oleh
`Persiapan_Data_Revised.ipynb`. Isinya dua kelompok:

- **Bagian A — Apa yang berubah setelah pembersihan.** Membuktikan bahwa data ini layak
  dipakai untuk analisis, bukan sekadar layak untuk model.
- **Bagian B — Siapa populasinya dan siapa yang berisiko.** Profil demografi,
  antropometri, prevalensi risiko per subkelompok, dan komorbiditas.

## Kelompok I-A — Apa yang Berubah Setelah Pembersihan

## I.1. Jejak Pembersihan per Tahap

In [ ]:

# ===================== WATERFALL PEMBERSIHAN =====================
# Dibaca dari file audit yang diekspor notebook Persiapan Data, supaya angkanya
# tidak dihitung ulang (satu sumber kebenaran).
audit = pd.read_csv(os.path.join(DATA_DIR, "audit_pembersihan.csv"))
audit["Selisih"] = audit["Baris tersisa"].diff().fillna(0).astype(int)
display(audit)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
bars = ax.bar(range(len(audit)), audit["Baris tersisa"], color="steelblue")
ax.set_xticks(range(len(audit)))
ax.set_xticklabels([t.split(". ", 1)[-1] for t in audit["Tahap"]],
                   rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Baris tersisa")
ax.set_title("Jumlah Baris di Setiap Tahap Pembersihan")

for i, (v, d) in enumerate(zip(audit["Baris tersisa"], audit["Selisih"])):
    ax.text(i, v, f"{v:,}" + (f"\n({d:+,})" if d else ""),
            ha="center", va="bottom", fontsize=8)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

hilang = df_raw.shape[0] - df.shape[0]
print(f"Total baris dibuang: {hilang:,} ({hilang / df_raw.shape[0] * 100:.2f}% dari data awal)")


Hanya sekitar 1% data yang dibuang. Angka ini penting untuk laporan: pembersihan di sini
**tidak** memangkas data secara agresif — sebagian besar perbaikan dilakukan dengan
membetulkan baris (menukar balik tinggi/berat), bukan membuangnya.


## I.2. Dampak Pembersihan pada Statistik Deskriptif

In [ ]:

# ===================== SEBELUM vs SESUDAH =====================
banding = pd.DataFrame({
    "SEBELUM — mean": [df_raw[c].mean() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SEBELUM — std": [df_raw[c].std() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SEBELUM — max": [df_raw[c].max() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SESUDAH — mean": [df[c].mean() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
    "SESUDAH — std": [df[c].std() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
    "SESUDAH — max": [df[c].max() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
}, index=["Usia", "Tinggi Badan", "Berat Badan", "IMT"]).round(2)

display(banding)


### Inilah alasan pembersihan bukan sekadar urusan modeling

Rata-rata IMT mentah adalah **69.5** — angka yang secara fisiologis mustahil sebagai
rata-rata populasi. Setelah dibersihkan, angkanya menjadi **22.3**, yang wajar untuk
populasi Indonesia.

Standar deviasi IMT mentah mencapai ribuan, akibat beberapa baris dengan tinggi badan
salah input. Artinya: **tanpa pembersihan, tidak ada satu pun statistik deskriptif dari
data ini yang bisa dilaporkan** — bukan hanya modelnya yang rusak, melainkan setiap
rata-rata, setiap simpangan baku, dan setiap grafik distribusi.

In [ ]:

# ===================== DISTRIBUSI SEBELUM vs SESUDAH =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

raw_cols = ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]
clean_cols = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]

for i, col in enumerate(raw_cols):
    lo, hi = df_raw[col].quantile([0.01, 0.99])
    sns.histplot(df_raw[col].clip(lo, hi), bins=40, kde=True,
                 ax=axes[0, i], color="lightcoral")
    axes[0, i].set_title(f"{col} — SEBELUM\n(dipotong p1-p99 agar terbaca)", fontsize=10)

for i, col in enumerate(clean_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=axes[1, i], color="mediumseagreen")
    axes[1, i].set_title(f"{col} — SESUDAH", fontsize=10)

plt.tight_layout()
plt.show()

## Kelompok I-B — Siapa Populasinya dan Siapa yang Berisiko

## I.3. Profil Demografi

In [ ]:

# ===================== KELOMPOK USIA =====================
AGE_BINS = [0, 5, 15, 25, 35, 45, 55, 65, 120]
AGE_LABELS = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["Kelompok_Usia"] = pd.cut(df["Usia"], bins=AGE_BINS, labels=AGE_LABELS, right=False)

ringkas = pd.DataFrame({
    "Jumlah": df["Kelompok_Usia"].value_counts().sort_index(),
    "%": (df["Kelompok_Usia"].value_counts(normalize=True).sort_index() * 100).round(2),
})
display(ringkas)

print(df["Jeniskelamin"].value_counts().to_string())
print()
print(df["segmen"].value_counts().to_string())

In [ ]:

# ===================== PIRAMIDA PENDUDUK & SEGMEN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Piramida penduduk ---
pyr = df.groupby(["Kelompok_Usia", "Jeniskelamin"], observed=True).size().unstack(fill_value=0)
axes[0].barh(pyr.index.astype(str), -pyr.get("L", 0), color="#4A90D9", label="Laki-laki")
axes[0].barh(pyr.index.astype(str), pyr.get("P", 0), color="#E8748C", label="Perempuan")
axes[0].set_title("Piramida Peserta Skrining")
axes[0].set_xlabel("Jumlah catatan skrining")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].legend()
axes[0].xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{abs(int(x)):,}")
)

# --- Segmen ---
seg = df["segmen"].value_counts()
axes[1].barh(seg.index[::-1], seg.values[::-1], color="steelblue")
axes[1].set_title("Distribusi Segmen Kepesertaan")
axes[1].set_xlabel("Jumlah catatan skrining")
for i, v in enumerate(seg.values[::-1]):
    axes[1].text(v, i, f" {v:,}", va="center", fontsize=8)

plt.tight_layout()
plt.show()


`segmen` adalah kategori kepesertaan BPJS, dan merupakan **satu-satunya proksi sosial
ekonomi** yang tersedia di data ini:

| Kode | Arti |
|---|---|
| APBN / APBD | Iuran ditanggung pemerintah pusat / daerah (umumnya kelompok kurang mampu) |
| PPU PN / PPU BU | Pekerja penerima upah — penyelenggara negara / badan usaha |
| PBPU | Pekerja bukan penerima upah (mandiri) |
| BP PN / BP SWASTA | Bukan pekerja (termasuk pensiunan) |

Karena itu, gradien prevalensi antar `segmen` di Bagian I.5 layak diperhatikan.


## I.4. Profil Antropometri & Status Gizi

In [ ]:

# ===================== IMT PER USIA & JENIS KELAMIN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.boxplot(data=df, x="Kelompok_Usia", y="IMT_recalculated",
            hue="Jeniskelamin", ax=axes[0], showfliers=False,
            palette={"L": "#4A90D9", "P": "#E8748C"})
axes[0].set_title("Distribusi IMT per Kelompok Usia & Jenis Kelamin")
axes[0].set_ylabel("IMT (kg/m2)")
axes[0].axhline(25, color="darkorange", ls="--", lw=1, label="Ambang overweight (25)")
axes[0].axhline(18.5, color="steelblue", ls="--", lw=1, label="Ambang underweight (18.5)")
axes[0].legend(fontsize=8)

# Prevalensi overweight+obese (dewasa saja -- kategori anak memakai standar berbeda)
dewasa = df[df["Usia"] > bs.CHILD_MAX_AGE]
ow = (dewasa.groupby(["Kelompok_Usia", "Jeniskelamin"], observed=True)["Kategori_BMI_WHO4"]
             .apply(lambda s: s.isin(["Overweight", "Obese"]).mean() * 100)
             .unstack())
ow.plot(kind="bar", ax=axes[1], color={"L": "#4A90D9", "P": "#E8748C"})
axes[1].set_title("Prevalensi Overweight + Obese (dewasa > 20 th)")
axes[1].set_ylabel("% dari kelompok")
axes[1].set_xlabel("Kelompok Usia")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== STATUS GIZI DEWASA =====================
gizi = pd.DataFrame({
    "Jumlah": dewasa["Kategori_BMI_WHO4"].value_counts(),
    "%": (dewasa["Kategori_BMI_WHO4"].value_counts(normalize=True) * 100).round(2),
}).reindex(["Underweight", "Normal", "Overweight", "Obese"])
display(gizi)

print(f"Populasi dewasa (> {bs.CHILD_MAX_AGE} th): {len(dewasa):,} catatan skrining")
print(f"Underweight       : {gizi.loc['Underweight', '%']:.1f}%")
print(f"Overweight + Obese: {gizi.loc[['Overweight', 'Obese'], '%'].sum():.1f}%")


Perhatikan bahwa **kekurangan gizi dan kelebihan gizi hadir bersamaan** dalam populasi yang
sama — pola yang dikenal sebagai *double burden of malnutrition*. Ini temuan kesehatan
masyarakat yang berdiri sendiri, terlepas dari model prediksi apa pun, dan berimplikasi
pada program intervensi yang perlu menyasar dua arah sekaligus.


## I.5. Prevalensi Risiko per Subkelompok

In [ ]:

# ===================== PREVALENSI PER TARGET =====================
def prevalensi(col):
    # % Berisiko di antara responden yang benar-benar ditanyai.
    s = df[col]
    ditanya = s.isin(["Berisiko", "Tidak Berisiko"])
    return df[ditanya], (s[ditanya] == "Berisiko")


ringkas = []
for col in TARGETS:
    sub, pos = prevalensi(col)
    ringkas.append({
        "Target": col.replace("Berisiko ", ""),
        "Rentang usia": ELIGIBLE[col],
        "N ditanyai": f"{len(sub):,}",
        "Berisiko": f"{int(pos.sum()):,}",
        "Prevalensi (%)": round(pos.mean() * 100, 2),
    })

display(pd.DataFrame(ringkas))
print("Prevalensi dihitung terhadap responden yang DITANYAI, bukan terhadap seluruh data.")

In [ ]:

# ===================== PREVALENSI per JENIS KELAMIN & USIA =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

for j, col in enumerate(TARGETS):
    sub, pos = prevalensi(col)
    sub = sub.assign(_pos=pos.values)

    # --- per jenis kelamin ---
    g = sub.groupby("Jeniskelamin")["_pos"].mean() * 100
    axes[0, j].bar(g.index, g.values, color=["#4A90D9", "#E8748C"])
    axes[0, j].set_title(f"{col.replace('Berisiko ', '')}\n(usia {ELIGIBLE[col]})", fontsize=10)
    axes[0, j].set_ylabel("% Berisiko")
    for i, v in enumerate(g.values):
        axes[0, j].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
    axes[0, j].margins(y=0.15)

    # --- per kelompok usia ---
    g2 = sub.groupby("Kelompok_Usia", observed=True)["_pos"].mean() * 100
    g2 = g2.dropna()
    axes[1, j].plot(g2.index.astype(str), g2.values, marker="o", color="crimson")
    axes[1, j].set_ylabel("% Berisiko")
    axes[1, j].set_xlabel("Kelompok Usia")
    axes[1, j].tick_params(axis="x", rotation=45)
    axes[1, j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## I.6. Komorbiditas — Berapa Risiko yang Ditanggung Sekaligus?

Analisis ini **hanya bermakna pada usia 45+**, karena hanya pada kelompok itulah keempat
pertanyaan sama-sama ditanyakan. Di bawah usia 45, Kanker Paru tidak ditanyakan sama
sekali, sehingga "tidak berisiko kanker paru" pada anak muda berarti *tidak diketahui*,
bukan *negatif*.

In [ ]:

# ===================== KOMORBIDITAS (USIA 45+) =====================
komorbid = df[df["Usia"] >= 45].copy()
for col in TARGETS:
    komorbid[col] = komorbid[col].map({"Tidak Berisiko": 0, "Berisiko": 1})
komorbid = komorbid.dropna(subset=TARGETS)

print(f"Populasi usia 45+ dengan keempat pertanyaan terjawab: {len(komorbid):,}")

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Heatmap ko-okurensi ---
short = [c.replace("Berisiko ", "") for c in TARGETS]
co = komorbid[TARGETS].corr()
co.index = co.columns = short
sns.heatmap(co, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, ax=axes[0])
axes[0].set_title("Korelasi Antar Risiko (usia 45+)")

# --- Jumlah risiko yang ditanggung ---
n_risk = komorbid[TARGETS].sum(axis=1).astype(int)
vc = n_risk.value_counts().sort_index()
axes[1].bar(vc.index.astype(str), vc.values, color="steelblue")
axes[1].set_title("Jumlah Risiko yang Ditanggung Bersamaan (usia 45+)")
axes[1].set_xlabel("Banyaknya risiko")
axes[1].set_ylabel("Jumlah catatan skrining")
for i, v in enumerate(vc.values):
    axes[1].text(i, v, f"{v:,}\n({v/len(komorbid)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=8)
axes[1].margins(y=0.18)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== KOMBINASI RISIKO TERBANYAK =====================
kombinasi = (
    komorbid[TARGETS].astype(int)
    .apply(lambda r: " + ".join([s for s, v in zip(short, r) if v]) or "(tidak ada)", axis=1)
    .value_counts()
    .head(10)
)
display(pd.DataFrame({
    "Jumlah": kombinasi,
    "%": (kombinasi / len(komorbid) * 100).round(2),
}))


## I.7. Subkelompok dengan Risiko Tertinggi

Tabel ini yang paling langsung bisa dipakai puskesmas: kombinasi kelompok usia, jenis
kelamin, dan segmen mana yang paling perlu diprioritaskan untuk tindak lanjut.

In [ ]:

# ===================== SUBKELOMPOK RISIKO TERTINGGI =====================
MIN_N = 300      # ambang minimal agar persentase tidak menyesatkan

def subkelompok_teratas(col, k=10):
    sub, pos = prevalensi(col)
    sub = sub.assign(_pos=pos.values)
    g = (sub.groupby(["Kelompok_Usia", "Jeniskelamin", "segmen"], observed=True)
            .agg(N=("_pos", "size"), Prevalensi=("_pos", "mean")))
    g = g[g["N"] >= MIN_N].copy()
    g["Prevalensi (%)"] = (g["Prevalensi"] * 100).round(2)
    return g.drop(columns="Prevalensi").sort_values("Prevalensi (%)", ascending=False).head(k)


for col in TARGETS:
    print("=" * 78)
    print(f"{col}   (usia {ELIGIBLE[col]}, minimal N={MIN_N})")
    print("=" * 78)
    display(subkelompok_teratas(col))


## I.8. Ringkasan

In [ ]:

# ===================== RINGKASAN AKHIR =====================
print("=" * 74)
print("RINGKASAN — POPULASI SKRINING BANTUL 2025 (data bersih)")
print("=" * 74)
print(f"Catatan skrining      : {len(df):,}")
print(f"Rentang usia          : {int(df['Usia'].min())} - {int(df['Usia'].max())} tahun")
print(f"Komposisi jenis kelamin: "
      f"{(df['Jeniskelamin'] == 'L').mean()*100:.1f}% L / {(df['Jeniskelamin'] == 'P').mean()*100:.1f}% P")
print(f"IMT rata-rata         : {df['IMT_recalculated'].mean():.2f} "
      f"(mentah: {df_raw['Index Masa Tubuh'].mean():.1f})")
print()
print("Prevalensi risiko (di antara yang ditanyai):")
for col in TARGETS:
    sub, pos = prevalensi(col)
    print(f"  {col.replace('Berisiko ', ''):<20} {pos.mean()*100:>6.2f}%   "
          f"(usia {ELIGIBLE[col]}, n={len(sub):,})")


# Bagian II — Klasifikasi BMI

Bagian ini mendalami **klasifikasi status gizi berdasarkan Indeks Massa Tubuh (IMT/BMI)**
pada data yang sudah dibersihkan, memakai dua standar yang berbeda menurut kelompok usia.

Seluruh logika klasifikasi berada di modul bersama `bmi_standards.py`, bukan disalin ulang
di sini, supaya definisinya tidak tercecer antar bagian.

## Standar yang dipakai

Sumber tabel: <https://www.calculator.net/bmi-calculator.html>

| Kelompok usia | Standar | Alasan |
|---|---|---|
| **> 20 tahun** | WHO (dewasa), 8 kategori | Tabel WHO dinyatakan berlaku untuk usia 20 tahun ke atas |
| **2 - 20 tahun** | CDC BMI-for-age percentile, per jenis kelamin | IMT anak berubah menurut usia & jenis kelamin, sehingga ambang tetap tidak berlaku |
| **< 2 tahun** | *tidak diklasifikasi* | BMI-for-age tidak berlaku secara medis; WHO merekomendasikan *Weight-for-Length* |

**Catatan batas usia.** Pada sumbernya, tabel WHO dinyatakan berlaku untuk usia ">= 20"
sedangkan tabel CDC untuk "2-20", sehingga **usia 20 tercakup oleh keduanya**. Proyek ini
memilih memasukkan usia 20 ke kelompok anak (CDC), dan batas ini dikendalikan satu
konstanta `bs.CHILD_MAX_AGE` agar mudah diubah bila rujukan PKL menentukan lain.

**Standar Indonesia (Kemenkes/IMT) tidak diimplementasikan** karena tidak ditemukan sumber
yang dapat disitasi pada saat penyusunan. Menuliskan ambang batas medis tanpa rujukan yang
jelas berisiko untuk laporan akademik. Struktur modul memungkinkan penambahannya nanti
tanpa mengubah bagian ini.


## II.1. Tabel Rujukan

In [ ]:

# ===================== TABEL WHO (DEWASA) =====================
who = []
prev = 0.0
for upper, label in bs.WHO_ADULT_BINS:
    rentang = f"< {upper:g}" if prev == 0 else (f">= {prev:g}" if np.isinf(upper) else f"{prev:g} - {upper:g}")
    who.append({"Kategori WHO": label, "Rentang IMT (kg/m2)": rentang,
                "Ringkas (WHO-4)": bs.WHO8_TO_WHO4[label]})
    prev = upper

display(pd.DataFrame(who))

In [ ]:

# ===================== TABEL CDC (ANAK 2-20 TAHUN) =====================
print("Kategori CDC berdasarkan persentil BMI-for-age:")
display(pd.DataFrame([
    {"Kategori": bs.CDC_CHILD_LABELS["underweight"], "Rentang persentil": "< 5%"},
    {"Kategori": bs.CDC_CHILD_LABELS["healthy"], "Rentang persentil": "5% - 85%"},
    {"Kategori": bs.CDC_CHILD_LABELS["at_risk"], "Rentang persentil": "85% - 95%"},
    {"Kategori": bs.CDC_CHILD_LABELS["overweight"], "Rentang persentil": "> 95%"},
]))

print("\nTabel ambang persentil per usia & jenis kelamin (1=L, 2=P):")
display(bs.CDC_BMI_REF.pivot(index="Usia", columns="Sex", values=["P5", "P85", "P95"]))


Perhatikan bagaimana ambangnya **bergerak menurut usia**. Untuk anak laki-laki, ambang
overweight (P95) naik dari sekitar 18.8 pada usia 2 tahun ke sekitar 30.6 pada usia 20
tahun. Inilah sebabnya ambang dewasa yang tetap (25 / 30) tidak boleh diterapkan pada
anak — anak usia 8 tahun dengan IMT 21 sudah tergolong overweight menurut CDC, padahal
menurut ambang dewasa ia masih "Normal".


## II.2. Hasil Klasifikasi

In [ ]:

# ===================== DISTRIBUSI KATEGORI =====================
# Kolom sudah dihitung di notebook Persiapan Data memakai modul yang sama.
dist = pd.DataFrame({
    "Jumlah": df["Kategori_BMI_WHO8"].value_counts(),
    "%": (df["Kategori_BMI_WHO8"].value_counts(normalize=True) * 100).round(2),
})
display(dist)

# Verifikasi: tidak boleh ada label yang dipakai lintas kelompok usia.
rentang = df.groupby("Kategori_BMI_WHO8")["Usia"].agg(["min", "max", "size"])
rentang.columns = ["Usia min", "Usia maks", "N"]
display(rentang.sort_values("Usia min"))


Tabel rentang usia di atas berfungsi sebagai **uji regresi**: setiap label hanya boleh
muncul pada satu kelompok usia. Label berawalan `Anak_` harus berhenti di usia 20, label
WHO harus mulai dari 21, dan `Balita (0-2th)` harus berhenti di usia 1.

Versi sebelumnya menulis label dewasa dan label anak ke dalam satu kolom **tanpa awalan**,
sehingga `Overweight` berarti dua hal berbeda sekaligus. Akibatnya kolom kategori BMI
diam-diam berfungsi sebagai penanda kelompok usia, dan setiap angka *feature importance*
yang dihitung darinya menjadi tidak sahih.

In [ ]:

# ===================== DISTRIBUSI PER KELOMPOK =====================
dewasa = df[df["Usia"] > bs.CHILD_MAX_AGE].copy()
anak = df[df["Usia"].between(bs.CHILD_MIN_AGE, bs.CHILD_MAX_AGE)].copy()
balita = df[df["Usia"] < bs.CHILD_MIN_AGE].copy()

print(f"Dewasa (> {bs.CHILD_MAX_AGE} th) : {len(dewasa):,}")
print(f"Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th)      : {len(anak):,}")
print(f"Balita (< {bs.CHILD_MIN_AGE} th)  : {len(balita):,}")

ORDER8 = [l for _, l in bs.WHO_ADULT_BINS]

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

d1 = dewasa["Kategori_BMI_WHO8"].value_counts().reindex(ORDER8).fillna(0)
axes[0].bar(range(len(d1)), d1.values, color="steelblue")
axes[0].set_xticks(range(len(d1)))
axes[0].set_xticklabels(d1.index, rotation=35, ha="right", fontsize=8)
axes[0].set_title(f"Kategori WHO — Dewasa (> {bs.CHILD_MAX_AGE} th, n={len(dewasa):,})")
axes[0].set_ylabel("Jumlah")
for i, v in enumerate(d1.values):
    axes[0].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=7)
axes[0].margins(y=0.15)

d2 = anak["Kategori_BMI_WHO8"].value_counts()
axes[1].bar(range(len(d2)), d2.values, color="mediumseagreen")
axes[1].set_xticks(range(len(d2)))
axes[1].set_xticklabels(d2.index, rotation=35, ha="right", fontsize=8)
axes[1].set_title(f"Kategori CDC — Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th, n={len(anak):,})")
axes[1].set_ylabel("Jumlah")
for i, v in enumerate(d2.values):
    axes[1].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=7)
axes[1].margins(y=0.15)

plt.tight_layout()
plt.show()


## II.3. WHO-8 vs WHO-4 — Dampak Peringkasan Kategori

Dataset menyediakan dua tingkat granularitas. Bagian ini memperlihatkan apa yang hilang
saat 8 kategori diringkas menjadi 4.

In [ ]:

# ===================== CROSSTAB WHO8 -> WHO4 =====================
ct = pd.crosstab(dewasa["Kategori_BMI_WHO8"], dewasa["Kategori_BMI_WHO4"])
ct = ct.reindex(index=[c for c in ORDER8 if c in ct.index])
display(ct)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
who4_order = ["Underweight", "Normal", "Overweight", "Obese"]
bottom = np.zeros(len(who4_order))
palette = plt.cm.viridis(np.linspace(0.15, 0.9, len(ORDER8)))

for lab, colr in zip(ORDER8, palette):
    if lab not in ct.index:
        continue
    vals = [ct.loc[lab, c] if c in ct.columns else 0 for c in who4_order]
    ax.bar(who4_order, vals, bottom=bottom, label=lab, color=colr)
    bottom += np.array(vals, dtype=float)

ax.set_title("Komposisi Kategori WHO-4 menurut Kategori WHO-8 (dewasa)")
ax.set_ylabel("Jumlah")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


Peringkasan menggabungkan tiga tingkat *thinness* menjadi satu "Underweight", dan tiga
kelas obesitas menjadi satu "Obese". Untuk pemodelan, versi ringkas biasanya cukup dan
mengurangi jumlah kolom. Untuk pelaporan kesehatan masyarakat, versi 8 kategori lebih
informatif karena **Obese Class III** menuntut penanganan yang berbeda dari
**Obese Class I**.


## II.4. Kelompok Balita (< 2 Tahun)

Kelompok ini **dilaporkan, bukan disembunyikan**. Mereka tetap ada di dataset dan tetap
dipakai untuk pemodelan TBC, hanya saja kategori BMI-nya sengaja dikosongkan.

In [ ]:

# ===================== PROFIL BALITA =====================
print(f"Jumlah balita (< {bs.CHILD_MIN_AGE} th) : {len(balita):,}")
print(f"Porsi dari seluruh data     : {len(balita)/len(df)*100:.2f}%")
print("(dataset TBC mencakup semua usia, jadi porsinya sama; target lain")
print(" punya batas usia minimal 15 atau 45 th sehingga tidak memuat balita)")
print()
display(balita[["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]].describe().round(2))

# Apakah risiko TBC pada balita cukup berarti untuk dipertahankan?
tbc_balita = (balita["Berisiko TBC"] == "Berisiko").mean() * 100
tbc_anak = (df[df["Usia"].between(2, 14)]["Berisiko TBC"] == "Berisiko").mean() * 100
tbc_semua = (df["Berisiko TBC"] == "Berisiko").mean() * 100
print(f"\nPrevalensi risiko TBC:")
print(f"  Balita (< 2 th) : {tbc_balita:.2f}%  (n={len(balita):,})")
print(f"  Anak (2-14 th)  : {tbc_anak:.2f}%")
print(f"  Seluruh populasi: {tbc_semua:.2f}%")


### Mengapa balita tetap dipertahankan tanpa kategori BMI

Tinggi dan berat mereka wajar sebagai antropometri bayi (median sekitar 71 cm / 8 kg), dan
prevalensi risiko TBC-nya **sebanding dengan kelompok anak yang lebih tua**. Jadi mereka
adalah kelompok yang nyata dan relevan secara klinis — TBC pada bayi justru berbahaya.

Yang tidak tersedia hanyalah **kategori BMI-nya**, karena BMI-for-age memang tidak berlaku
di bawah usia 2 tahun. Ada tiga pilihan, dan hanya satu yang bisa dipertanggungjawabkan:

| Pilihan | Penilaian |
|---|---|
| Buang seluruh barisnya | Membuang ratusan bayi berisiko TBC, dan membuat model TBC diam-diam tidak berlaku untuk kelompok termuda |
| Paksakan ke kategori usia 2 tahun | Mengarang klasifikasi klinis yang standarnya sendiri menyatakan tidak berlaku |
| **Pertahankan, kosongkan kategorinya** | **Dipilih.** Baris tetap menyumbang usia, jenis kelamin, tinggi, berat, dan segmen; hanya fitur turunan yang absen |

Kehilangan kategori BMI untuk kelompok ini praktis tidak merugikan model, karena kategori
BMI memang redundan terhadap `IMT_recalculated` yang tetap tersedia.


## II.5. Kategori BMI vs Risiko Penyakit

In [ ]:

# ===================== PREVALENSI RISIKO per KATEGORI BMI =====================
# Dibatasi ke dewasa supaya ambang yang dipakai konsisten.
hasil = {}
for col in TARGETS:
    sub = dewasa[dewasa[col].isin(["Berisiko", "Tidak Berisiko"])]
    if sub.empty:
        continue
    hasil[col.replace("Berisiko ", "")] = (
        sub.assign(_p=sub[col].eq("Berisiko"))
           .groupby("Kategori_BMI_WHO8")["_p"].mean() * 100
    ).round(2)

tab = pd.DataFrame(hasil).reindex([c for c in ORDER8])
display(tab)

fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)
for ax, col in zip(axes.flat, tab.columns):
    vals = tab[col].dropna()
    ax.barh(range(len(vals)), vals.values, color="steelblue")
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(vals.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(col)
    ax.set_xlabel("% Berisiko")
    for i, v in enumerate(vals.values):
        ax.text(v, i, f" {v:.0f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== APAKAH LABELNYA SEKADAR ATURAN IMT? =====================
rows = []
for col in TARGETS:
    sub = dewasa[dewasa[col].isin(["Berisiko", "Tidak Berisiko"])]
    if sub.empty:
        continue
    pos = sub[col].eq("Berisiko")
    hi = sub["IMT_recalculated"] > 25
    rows.append({
        "Target": col.replace("Berisiko ", ""),
        "N dewasa": f"{len(sub):,}",
        "P(Berisiko | IMT > 25)": round(pos[hi].mean(), 4),
        "P(Berisiko | IMT <= 25)": round(pos[~hi].mean(), 4),
        "Pengecualian di IMT > 25": int((~pos[hi]).sum()),
        "Selisih": round(pos[hi].mean() - pos[~hi].mean(), 4),
    })

display(pd.DataFrame(rows))


### Temuan — label Diabetes praktis merupakan aturan `IMT > 25`

Grafik dan tabel di atas memperlihatkan **fungsi tangga**, bukan gradien: pada responden
dewasa, hampir setiap orang ber-IMT di atas 25 diberi label `Berisiko Diabetes Melitus`,
dengan jumlah pengecualian yang sangat kecil.

Hipertensi menunjukkan hubungan kuat tetapi **tidak** deterministik. Kanker Paru dan TBC
praktis tidak terpengaruh IMT sama sekali — selisih prevalensinya mendekati nol.

**Konsekuensi yang harus dibawa ke laporan:**

1. **Prevalensi "berisiko diabetes" bukan temuan epidemiologis yang berdiri sendiri.**
   Sebagian besar hanyalah pernyataan ulang dari prevalensi overweight pada populasi ini.
   Melaporkannya sebagai ukuran risiko diabetes yang independen akan menyesatkan.
2. **Model diabetes yang memakai IMT sebagai fitur sedang merekonstruksi aturan penilaian
   kuesioner**, bukan memprediksi penyakit. Akurasi tinggi pada target ini tidak boleh
   dibaca sebagai keberhasilan prediksi.
3. Untuk Kanker Paru dan TBC, kategori BMI **hampir tidak membawa informasi**. Faktor yang
   relevan di sana adalah jenis kelamin (Kanker Paru) serta usia dan segmen (TBC).


## II.6. Ringkasan

In [ ]:

# ===================== RINGKASAN =====================
print("=" * 72)
print("RINGKASAN KLASIFIKASI BMI")
print("=" * 72)
print(f"Sumber standar : {bs.SOURCE_URL}")
print()
print(f"Dewasa (> {bs.CHILD_MAX_AGE} th)  : {len(dewasa):>7,}  -> WHO, 8 kategori")
print(f"Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th)       : {len(anak):>7,}  -> CDC BMI-for-age percentile")
print(f"Balita (< {bs.CHILD_MIN_AGE} th)   : {len(balita):>7,}  -> tidak diklasifikasi")
print(f"{'TOTAL':<20} {len(df):>7,}")
print()
print("Kategori yang dihasilkan:")
print(f"  Kategori_BMI_WHO8 : {df['Kategori_BMI_WHO8'].nunique()} nilai unik")
print(f"  Kategori_BMI_WHO4 : {df['Kategori_BMI_WHO4'].nunique()} nilai unik")


# Bagian III — Hubungan Antar Variabel

Bagian ini menjawab satu pertanyaan:

> **Hubungan apa yang ada antara karakteristik seseorang dan risiko penyakitnya, dan
> seberapa kuat hubungan itu?**

Analisisnya murni **deskriptif** — mengukur dan menggambarkan hubungan. Tidak ada pemodelan
di sini; itu ada di `Script penyakitan/Stacked Revisi.ipynb`.

## Susunan bagian ini

Bagian ini disusun **menurut penyakit**, bukan menurut variabel. Tiap penyakit mendapat satu
profil lengkap berisi seluruh variabel bebasnya sekaligus, sehingga pertanyaan *"apa yang
menggerakkan penyakit ini"* terjawab dalam satu tampilan.

| Bagian | Isi |
|---|---|
| III.1 | Data & fungsi bantu |
| III.2 | **Hubungan antar variabel bebas** — sebelum masuk ke penyakit |
| III.3 | **Diabetes Melitus** |
| III.4 | **Hipertensi** |
| III.5 | **Kanker Paru** |
| III.6 | **TBC** |
| III.7 | **Segmen** — peta panas seluruh penyakit |
| III.8 | Ringkasan lintas penyakit |
| — | Penutup ketiga bagian ada di **Kesimpulan**; metode & cara membaca grafik di **Lampiran A** |

## Yang sudah dibahas di Bagian I & II (tidak diulang di sini)

- Korelasi **antar target** (komorbiditas) — **Bagian I.6**
- Prevalensi per kategori BMI — **Bagian II.5**
- Prevalensi per subkelompok usia/jenis kelamin/segmen — **Bagian I.5**
- *Feature importance* berbasis model — `Stacked Revisi` §7 (notebook terpisah)

Ukuran statistik yang dipakai, panduan membaca besarannya, dan alasan bagian ini
tidak memuat nilai-p ada di **Lampiran A** di akhir notebook.


## III.1. Data & Fungsi Bantu

## Dua aturan yang berlaku di seluruh Bagian III

**1. Tiap target hanya boleh dihitung pada populasi yang berhak ditanyai.** Label
`Tidak Dapat Pertanyaan` berarti responden **tidak memenuhi syarat usia**, bukan datanya
hilang. Diabetes & Hipertensi hanya ditanyakan pada usia 15+, Kanker Paru 45+, TBC semua
usia. Menghitung korelasi tanpa menyaring ini akan mencampur "tidak berisiko" dengan
"tidak ditanya".

**2. Kekuatan hubungan antar penyakit tidak sepenuhnya sebanding.** Karena populasinya
berbeda, angka untuk Kanker Paru (hanya 45+) tidak bisa dibandingkan mentah-mentah dengan
TBC (semua usia). Ini diingatkan lagi di III.8.

In [ ]:

# ===================== FUNGSI UKURAN ASOSIASI =====================
def eligible(nama):
    # Penyaringan eligibilitas hanya ditulis sekali, di prevalensi() pada Bagian I.
    # Di sini bentuknya saja yang disesuaikan: dikunci nama pendek, target 0/1.
    col, _ = TARGET_INFO[nama]
    sub, pos = prevalensi(col)
    return sub, pos.astype(int)


def cramers_v(x, y):
    # Kekuatan asosiasi antara dua variabel kategorik (0..1, tanpa arah).
    # Untuk tabel 2x2 nilainya sama dengan koefisien phi.
    ct = pd.crosstab(x, y)
    if min(ct.shape) < 2:
        return np.nan
    n = ct.to_numpy().sum()
    exp = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / n
    chi2 = ((ct.to_numpy() - exp) ** 2 / exp).sum()
    return float(np.sqrt((chi2 / n) / (min(ct.shape) - 1)))


def correlation_ratio(kategori, nilai):
    # Rasio korelasi eta (0..1): akar dari proporsi ragam 'nilai' yang dijelaskan
    # oleh perbedaan rata-rata antar kategori.
    grand = nilai.mean()
    g = nilai.groupby(kategori, observed=True)
    antar = (g.count() * (g.mean() - grand) ** 2).sum()
    total = ((nilai - grand) ** 2).sum()
    return float(np.sqrt(antar / total)) if total > 0 else np.nan


def tafsir(v):
    a = abs(v)
    if np.isnan(a):
        return "-"
    if a < 0.10:
        return "diabaikan"
    if a < 0.30:
        return "lemah"
    if a < 0.50:
        return "sedang"
    return "kuat"


print("Fungsi siap: eligible(), cramers_v(), correlation_ratio(), tafsir()")

## III.2. Hubungan Antar Variabel Bebas

Sebelum melihat penyakit, perlu dipahami dulu bagaimana variabel-variabel bebas saling
berkaitan. Kalau dua variabel sangat berkorelasi satu sama lain, keduanya membawa informasi
yang tumpang tindih — dan sebuah variabel bisa tampak berhubungan dengan penyakit padahal
sebenarnya hanya menumpang pada variabel lain.

In [ ]:

# ===================== MATRIKS KORELASI NUMERIK =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

for ax, metode, judul in [(axes[0], "pearson", "Pearson (hubungan linear)"),
                          (axes[1], "spearman", "Spearman (hubungan monoton)")]:
    sns.heatmap(df[NUM].corr(method=metode), annot=True, fmt=".3f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, ax=ax, annot_kws={"fontsize": 12},
                cbar_kws={"shrink": 0.8})
    ax.set_title(judul)

fig.suptitle("Korelasi Antar Variabel Numerik — SEMUA USIA (0-113)", y=1.00)
plt.tight_layout()
plt.show()


Tiga hal langsung terlihat, dan **dua di antaranya menyesatkan** kalau dibaca apa adanya:

1. **`Berat Badan` ↔ `IMT` = 0.839** — wajar dan tidak informatif: IMT dihitung *dari* berat
   badan. Keduanya bukan dua informasi terpisah.
2. **`Tinggibadan` ↔ `Berat Badan` = 0.797** — terlihat sangat kuat.
3. **`Usia` ↔ `IMT` = 0.356** — terlihat sedang.

Poin 2 dan 3 dihitung pada **seluruh rentang usia 0–113 tahun**, dan populasi itu memuat
anak-anak. Anak yang bertambah usia akan bertambah tinggi, bertambah berat, dan bertambah IMT
sekaligus — bukan karena ketiganya saling memengaruhi, melainkan karena semuanya mengikuti
**pertumbuhan**.

In [ ]:

# ===================== ETA: KATEGORIK -> NUMERIK (DEWASA) =====================
# Dibatasi ke dewasa supaya efek pertumbuhan tidak mencemari hasilnya.
baris = []
for c in CAT:
    for n in NUM:
        baris.append({"Kategorik": c, "Numerik": n,
                      "eta": round(correlation_ratio(dewasa[c], dewasa[n]), 3)})
eta_tab = pd.DataFrame(baris).pivot(index="Kategorik", columns="Numerik", values="eta")[NUM]

print("Rasio korelasi eta (0..1) — dewasa >20 tahun:")
display(eta_tab)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.boxplot(data=dewasa, x="Jeniskelamin", y="Tinggibadan", ax=axes[0],
            hue="Jeniskelamin", legend=False, showfliers=False,
            palette=vz.PALETTE["SEX"])
axes[0].set_title(f"Tinggi Badan menurut Jenis Kelamin\n"
                  f"(eta = {eta_tab.loc['Jeniskelamin','Tinggibadan']:.3f})")

urut = dewasa.groupby("segmen")["Usia"].median().sort_values().index
sns.boxplot(data=dewasa, x="segmen", y="Usia", order=urut, ax=axes[1],
            color="steelblue", showfliers=False)
axes[1].set_title(f"Usia menurut Segmen\n(eta = {eta_tab.loc['segmen','Usia']:.3f})")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


### Temuan: dua perancu yang akan muncul lagi nanti

- **Jenis kelamin → Tinggi badan, η ≈ 0.58.** Kuat dan sepenuhnya terduga secara biologis.
  Konsekuensinya: `Tinggibadan` sebagian berfungsi sebagai **penanda jenis kelamin**. Ini
  ditagih kembali di III.5 (Kanker Paru).
- **Segmen → Usia, η ≈ 0.42.** Segmen kepesertaan BPJS sangat berkaitan dengan tahap hidup —
  misalnya segmen "bukan pekerja" banyak berisi pensiunan. Ini ditagih kembali di III.6 (TBC).

Keduanya berarti: bila nanti tinggi badan atau segmen tampak berhubungan dengan penyakit,
sebagian hubungan itu bisa jadi milik jenis kelamin atau usia.

## Fungsi penggambar profil penyakit

Satu fungsi dipakai untuk keempat penyakit agar III.3–III.6 tidak mengulang kode dan seluruh
gambar dijamin memakai perlakuan yang sama persis.

In [ ]:

# ===================== FUNGSI PROFIL PENYAKIT =====================
def profil_penyakit(nama, n_desil=10):
    # Menggambar satu gambar berisi 9 panel:
    #   baris 1 (4 panel) : histogram densitas Berisiko vs Tidak Berisiko
    #   baris 2 (4 panel) : garis prevalensi per desil
    #   kolom 5           : batang prevalensi menurut jenis kelamin (membentang 2 baris)
    # Mengembalikan tabel ringkas asosiasi untuk penyakit ini.
    sub, y = eligible(nama)
    base = y.mean()
    usia_lbl = TARGET_INFO[nama][1]

    fig = plt.figure(figsize=(27, 11))
    gs = gridspec.GridSpec(2, 5, figure=fig, width_ratios=[1, 1, 1, 1, 0.7],
                           hspace=0.32, wspace=0.28)

    hasil = {}
    for j, var in enumerate(NUM):
        # ---------- baris 1: histogram densitas ----------
        ax_h = fig.add_subplot(gs[0, j])
        lo, hi = sub[var].quantile([0.005, 0.995])      # potong ekor agar terbaca
        bins = np.linspace(lo, hi, 45)
        for kelas, lbl in [(0, "Tidak Berisiko"), (1, "Berisiko")]:
            ax_h.hist(sub.loc[y == kelas, var], bins=bins, density=True, alpha=0.55,
                      color=vz.PALETTE["RISK"][lbl], label=lbl)
        ax_h.set_title(var)
        ax_h.set_ylabel("kepadatan" if j == 0 else "")
        ax_h.set_xlim(lo, hi)
        if j == 0:
            ax_h.legend(fontsize=9)

        # ---------- baris 2: garis prevalensi per desil ----------
        ax_l = fig.add_subplot(gs[1, j])
        q = pd.qcut(sub[var], n_desil, duplicates="drop")
        prev = y.groupby(q, observed=True).mean() * 100
        # Titik diletakkan pada MEDIAN nilai di dalam tiap desil, bukan titik tengah
        # intervalnya. Desil pertama dan terakhir intervalnya sangat lebar (menjangkau
        # nilai ekstrem), sehingga titik tengahnya bisa jatuh di luar sumbu.
        xpos = sub.groupby(q, observed=True)[var].median()
        ax_l.plot(xpos.values, prev.values, marker="o", lw=2.5, color="#153763")
        ax_l.axhline(base * 100, color="crimson", ls="--",
                     label=f"rata-rata {base*100:.1f}%")
        r = sub[var].corr(y)
        hasil[var] = r
        ax_l.set_title(f"r = {r:+.3f}  [{tafsir(r)}]", fontsize=12)
        ax_l.set_xlabel(f"{var} (titik tengah desil)")
        ax_l.set_ylabel("% Berisiko" if j == 0 else "")
        ax_l.set_xlim(lo, hi)                            # sejajar dengan histogram di atas
        ax_l.legend(fontsize=9)

    # ---------- kolom 5: jenis kelamin ----------
    ax_s = fig.add_subplot(gs[:, 4])
    prev_s = y.groupby(sub["Jeniskelamin"], observed=True).mean() * 100
    ax_s.bar(prev_s.index, prev_s.values,
             color=[vz.PALETTE["SEX"][s] for s in prev_s.index])
    ax_s.axhline(base * 100, color="crimson", ls="--", label=f"rata-rata {base*100:.1f}%")
    v = cramers_v(sub["Jeniskelamin"], y)
    hasil["Jeniskelamin"] = v
    ax_s.set_title(f"Jenis Kelamin\nCramer's V = {v:.3f}  [{tafsir(v)}]")
    ax_s.set_ylabel("% Berisiko")
    ax_s.legend(fontsize=9)
    vz.annotate_bars(ax_s, fmt="{:.1f}%")

    lbl_usia = usia_lbl if usia_lbl == "semua usia" else f"usia {usia_lbl}"
    fig.suptitle(f"{nama}  —  {lbl_usia}  |  n = {len(sub):,}  |  "
                 f"prevalensi {base*100:.2f}%", y=0.99, fontsize=17)
    plt.show()

    ringkas = pd.DataFrame({"Ukuran": pd.Series(hasil).round(3)})
    ringkas["Tafsiran"] = ringkas["Ukuran"].map(tafsir)
    ringkas["Jenis ukuran"] = ["point-biserial r"] * len(NUM) + ["Cramer's V"]
    return ringkas.reindex(NUM + ["Jeniskelamin"])


print("Fungsi profil_penyakit() siap.")

## III.3. Diabetes Melitus

Berlaku untuk usia **15 tahun ke atas**.

In [ ]:

display(profil_penyakit("Diabetes"))


### Membaca profil Diabetes

**Baris atas** memperlihatkan pemisahan paling jelas di antara keempat penyakit: pada panel
`IMT_recalculated` dan `Berat Badan`, kurva merah (Berisiko) dan hijau (Tidak Berisiko)
hampir tidak bertumpang tindih. Kedua kelompok benar-benar menempati wilayah yang berbeda.

**Baris bawah** memperlihatkan bentuk yang tidak biasa pada IMT: bukan kenaikan bertahap,
melainkan **lompatan tegak** di sekitar IMT 25, dari belasan persen langsung ke hampir 100%.

| Variabel | *r* | Tafsiran |
|---|---|---|
| IMT | **+0.616** | kuat |
| Berat Badan | +0.515 | kuat |
| Usia | +0.207 | lemah |
| Tinggi Badan | −0.074 | dapat diabaikan |
| Jenis Kelamin | 0.075 | dapat diabaikan |

### Peringatan: angka 0.616 bukan temuan epidemiologis

Label "Berisiko Diabetes" pada kuesioner skrining praktis **diturunkan dari aturan
`IMT > 25`** — `P(Berisiko | IMT > 25) = 0.999`. Bentuk tangga pada grafik itu adalah tanda
aturan, bukan tanda biologi.

Jadi korelasi kuat antara IMT dan label diabetes bersifat **melingkar**: kita mengukur ulang
aturan yang dipakai untuk membuat labelnya. Buktinya lengkap ada di **Bagian II.5**.

Yang tersisa sebagai informasi nyata adalah **usia** (+0.207): di antara orang ber-IMT di
bawah 25, proporsi berisiko tetap naik seiring usia.

## III.4. Hipertensi

Berlaku untuk usia **15 tahun ke atas**.

In [ ]:

display(profil_penyakit("Hipertensi"))


### Membaca profil Hipertensi

Polanya mirip Diabetes tetapi **jauh lebih lemah**, dan bentuknya berbeda secara penting.

| Variabel | *r* | Tafsiran |
|---|---|---|
| IMT | **+0.312** | sedang |
| Berat Badan | +0.297 | lemah |
| Usia | +0.054 | dapat diabaikan |
| Tinggi Badan | +0.042 | dapat diabaikan |
| Jenis Kelamin | 0.035 | dapat diabaikan |

**Perbedaan bentuk itu penting.** Grafik IMT pada Hipertensi menanjak **mulus dan bertahap**
di sepanjang rentang — tidak ada lompatan di titik tertentu. Inilah bentuk yang diharapkan
dari hubungan biologis yang sesungguhnya, dan kontras jelas dengan bentuk tangga milik
Diabetes.

Dengan kata lain: Hipertensi memberi bukti bahwa IMT **memang** berkaitan dengan risiko
kesehatan — dan justru karena bentuknya mulus, hubungan ini lebih dapat dipercaya daripada
angka Diabetes yang jauh lebih besar.

**Usia hampir tidak berpengaruh** (+0.054), meski prevalensinya tinggi (60,7%). Hipertensi
banyak ditemukan di semua kelompok usia dewasa, bukan khusus pada lansia. Pada baris atas
terlihat kedua kurva usia hampir bertumpuk sempurna.

## III.5. Kanker Paru

Berlaku untuk usia **45 tahun ke atas**. Perhatikan bahwa rentang usianya sudah terpotong,
sehingga variasi usia yang tersisa kecil.

In [ ]:

display(profil_penyakit("Kanker Paru"))


### Membaca profil Kanker Paru

Profil penyakit ini **sepenuhnya berbeda** dari dua sebelumnya.

| Variabel | Ukuran | Tafsiran |
|---|---|---|
| **Jenis Kelamin** | **0.317** | **sedang — tertinggi** |
| Tinggi Badan | +0.203 | lemah |
| Berat Badan | +0.080 | dapat diabaikan |
| IMT | −0.032 | dapat diabaikan |
| Usia | −0.031 | dapat diabaikan |

Panel jenis kelamin di kolom kanan langsung menjelaskan semuanya: **22,3% pada laki-laki
melawan 2,4% pada perempuan** — selisih hampir sepuluh kali lipat.

IMT dan usia praktis tidak berpengaruh; pada baris atas, kurva merah dan hijau untuk kedua
variabel itu hampir bertumpuk sempurna.

Tetapi ada satu angka yang mencurigakan: **Tinggi Badan +0.203**. Secara biologis tidak ada
alasan tinggi badan memengaruhi risiko kanker paru. Bagian berikut mengujinya.

In [ ]:

# ===================== UJI: APAKAH TINGGI BADAN HANYA PROKSI JENIS KELAMIN? =====================
sub_k, y_k = eligible("Kanker Paru")
sub_k = sub_k.assign(_y=y_k.values)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Kiri: prevalensi vs tinggi, gabungan dan dipisah jenis kelamin ---
q_all = pd.qcut(sub_k["Tinggibadan"], 10, duplicates="drop")
prev_all = sub_k.groupby(q_all, observed=True)["_y"].mean() * 100
x_all = sub_k.groupby(q_all, observed=True)["Tinggibadan"].median()
axes[0].plot(x_all.values, prev_all.values, marker="o", lw=3,
             color="black", label="gabungan (L+P)")

for s, lbl in [("L", "Laki-laki saja"), ("P", "Perempuan saja")]:
    g = sub_k[sub_k["Jeniskelamin"] == s]
    q = pd.qcut(g["Tinggibadan"], 10, duplicates="drop")
    prev = g.groupby(q, observed=True)["_y"].mean() * 100
    xg = g.groupby(q, observed=True)["Tinggibadan"].median()
    axes[0].plot(xg.values, prev.values, marker="o", lw=2.2,
                 color=vz.PALETTE["SEX"][s], label=lbl)

axes[0].set_title("Kanker Paru vs Tinggi Badan\ngabungan vs di dalam tiap jenis kelamin")
axes[0].set_xlabel("Tinggi Badan (titik tengah desil)")
axes[0].set_ylabel("% Berisiko")
axes[0].legend()

# --- Kanan: korelasi sebelum vs sesudah dikendalikan ---
korelasi = {"Gabungan\n(L+P)": sub_k["Tinggibadan"].corr(sub_k["_y"])}
for s, lbl in [("L", "Laki-laki\nsaja"), ("P", "Perempuan\nsaja")]:
    g = sub_k[sub_k["Jeniskelamin"] == s]
    korelasi[lbl] = g["Tinggibadan"].corr(g["_y"])

axes[1].bar(list(korelasi), list(korelasi.values()),
            color=["black", vz.PALETTE["SEX"]["L"], vz.PALETTE["SEX"]["P"]])
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Korelasi Tinggi Badan - Kanker Paru")
axes[1].set_ylabel("point-biserial r")
vz.annotate_bars(axes[1], fmt="{:+.3f}")

plt.tight_layout()
plt.show()

for k, v in korelasi.items():
    print(f"  {k.replace(chr(10), ' '):<22} r = {v:+.3f}")


### Temuan: hubungan Tinggi Badan–Kanker Paru **sepenuhnya** efek jenis kelamin

| Populasi | point-biserial *r* |
|---|---|
| Gabungan (L+P) | **+0.203** |
| Laki-laki saja | **+0.055** |
| Perempuan saja | **−0.011** |

Begitu jenis kelamin dikendalikan, hubungannya **lenyap**. Pada panel kiri terlihat jelas:
garis hitam (gabungan) menanjak, tetapi garis biru dan merah muda masing-masing **datar** —
dan keduanya berada pada ketinggian yang sangat berbeda.

Mekanismenya sederhana: laki-laki rata-rata lebih tinggi (η = 0.58 dari III.2), dan laki-laki
punya risiko Kanker Paru jauh lebih besar. Tinggi badan hanya **menandai** jenis kelamin.

Ini contoh perancu paling bersih di seluruh Bagian III: korelasi 0.203 yang terlihat cukup
berarti ternyata **nol** setelah dikoreksi.

**Konsekuensi untuk laporan:** jangan pernah menulis bahwa orang yang lebih tinggi berisiko
Kanker Paru lebih besar. Yang benar: laki-laki berisiko lebih besar, dan laki-laki kebetulan
lebih tinggi. Jenis kelamin sendiri kemungkinan besar adalah **penanda pengganti kebiasaan
merokok**, yang tidak tersedia di data.

## III.6. TBC

Berlaku untuk **semua usia (0–113)** — satu-satunya target tanpa batas usia. Justru karena
itu, TBC paling rawan terkena efek perancu usia.

In [ ]:

display(profil_penyakit("TBC"))


### Membaca profil TBC

| Variabel | Ukuran | Tafsiran |
|---|---|---|
| Usia | **+0.134** | lemah — tertinggi |
| Berat Badan | +0.070 | dapat diabaikan |
| IMT | +0.066 | dapat diabaikan |
| Tinggi Badan | +0.049 | dapat diabaikan |
| Jenis Kelamin | 0.002 | dapat diabaikan |

**Seluruh variabel lemah.** Nilai tertinggi hanya 0.134, dan sisanya di bawah ambang 0.10.
Tidak ada satu pun karakteristik yang menjelaskan risiko TBC dengan baik.

Pada baris atas, kurva merah dan hijau bertumpang tindih sangat besar di keempat variabel —
gambaran visual dari kesimpulan yang sama. Ini konsisten dengan performa model yang paling
rendah pada target ini: keterbatasannya ada pada **variabel yang tersedia**, bukan pada
metode.

Dua hubungan tetap perlu diuji karena TBC mencakup semua usia:

1. **TBC ↔ IMT** — grafik naik, padahal literatur justru mengaitkan TBC dengan gizi kurang
2. **TBC ↔ segmen** — segmen sangat terkait usia (η = 0.42 dari III.2)

In [ ]:

# ===================== UJI PERANCU 1: TBC vs IMT, DIKENDALIKAN USIA =====================
sub_b, y_b = eligible("TBC")
sub_b = sub_b.assign(_y=y_b.values)
band_b = pd.cut(sub_b["Usia"], [0, 15, 30, 45, 60, 120], right=False,
                labels=["0-14", "15-29", "30-44", "45-59", "60+"])
sub_b = sub_b.assign(_band=band_b)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

for b in sub_b["_band"].cat.categories:
    s = sub_b[sub_b["_band"] == b]
    if len(s) < 500:
        continue
    q = pd.qcut(s["IMT_recalculated"], 5, duplicates="drop")
    prev = s.groupby(q, observed=True)["_y"].mean() * 100
    xq = s.groupby(q, observed=True)["IMT_recalculated"].median()
    axes[0].plot(xq.values, prev.values, marker="o", label=f"usia {b}")

axes[0].set_title("TBC vs IMT — dipisah per kelompok usia")
axes[0].set_xlabel("IMT (titik tengah kuintil)")
axes[0].set_ylabel("% Berisiko TBC")
axes[0].legend(title="Kelompok usia")

kes = sub_b["IMT_recalculated"].corr(sub_b["_y"])
dalam = (sub_b.groupby("_band", observed=True)
              .apply(lambda g: g["IMT_recalculated"].corr(g["_y"]), include_groups=False))
axes[1].bar(dalam.index.astype(str), dalam.values, color="steelblue")
axes[1].axhline(kes, color="crimson", ls="--", label=f"gabungan semua usia ({kes:+.3f})")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Korelasi TBC-IMT: gabungan vs di dalam kelompok usia")
axes[1].set_xlabel("Kelompok usia")
axes[1].set_ylabel("point-biserial r")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Korelasi TBC-IMT gabungan semua usia : {kes:+.3f}")
for b, v in dalam.items():
    print(f"   usia {str(b):<7} {v:+.3f}")


### Temuan 1: sebagian efek usia, tetapi tidak seluruhnya

| Populasi | *r* |
|---|---|
| Gabungan semua usia | **+0.066** |
| Usia 0–14 | **−0.030** (berbalik tanda) |
| Usia 15–29 | +0.015 |
| Usia 30–44 | +0.040 |
| Usia 45–59 | +0.051 |
| Usia 60+ | +0.063 |

Pada **anak-anak hubungannya berbalik**, pada **dewasa muda praktis nol**, tetapi pada
**lansia nilainya hampir sama dengan angka gabungan**.

Jadi kesimpulan yang jujur bukan "seluruhnya efek usia", melainkan: sebagian kenaikan pada
grafik profil memang efek komposisi usia, tetapi pada kelompok dewasa tua tetap ada hubungan
positif lemah yang bertahan.

**Yang terpenting: seluruh nilainya dapat diabaikan.** Bahkan yang tertinggi (0.066) jauh di
bawah ambang 0.10. IMT bukan penjelas risiko TBC yang berarti.

In [ ]:

# ===================== UJI PERANCU 2: TBC vs SEGMEN, DIKENDALIKAN USIA =====================
piv = sub_b.groupby(["_band", "segmen"], observed=True)["_y"].agg(["mean", "size"])
piv = piv[piv["size"] >= 300]["mean"].unstack() * 100

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
sns.heatmap(piv, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
            annot_kws={"fontsize": 11}, cbar_kws={"label": "% Berisiko TBC"})
ax.set_title("Prevalensi TBC menurut Segmen DI DALAM Tiap Kelompok Usia\n"
             "(sel dengan n < 300 dikosongkan)")
ax.set_xlabel("Segmen")
ax.set_ylabel("Kelompok usia")
plt.tight_layout()
plt.show()

print(f"Cramer's V segmen-TBC, gabungan semua usia : {cramers_v(sub_b['segmen'], sub_b['_y']):.3f}")
for b in sub_b["_band"].cat.categories:
    s = sub_b[sub_b["_band"] == b]
    if len(s) >= 500:
        print(f"   usia {str(b):<7} {cramers_v(s['segmen'], s['_y']):.3f}")


### Temuan 2: hubungan segmen–TBC **bertahan**, dan justru menguat pada usia tua

| Populasi | Cramér's V |
|---|---|
| Gabungan semua usia | **0.094** |
| Usia 0–14 | 0.037 |
| Usia 15–29 | 0.030 |
| Usia 30–44 | 0.038 |
| Usia 45–59 | 0.062 |
| Usia 60+ | **0.096** |

Asosiasi segmen–TBC **menguat seiring usia**, dan pada kelompok 60+ bahkan sedikit melampaui
angka gabungan. Artinya hubungan ini **tidak runtuh** ketika usia dikendalikan — berbeda
dari IMT yang berbalik tanda pada anak-anak.

Segmen membawa informasi di luar sekadar usia, tetapi informasinya **hanya relevan pada
kelompok dewasa tua**. Karena segmen mencerminkan kategori kepesertaan BPJS — dan secara
tidak langsung kondisi sosial ekonomi — arah temuan ini sejalan dengan literatur yang
mengaitkan risiko TBC dengan faktor sosial ekonomi.

**Tetap perlu kejujuran soal besaran:** 0.096 masih tergolong "dapat diabaikan". Segmen
adalah variabel terkuat kedua untuk TBC, tetapi itu lebih menunjukkan **betapa lemahnya semua
variabel untuk TBC** daripada betapa kuatnya segmen.

## III.7. Segmen: Semua Penyakit

`segmen` adalah kategori kepesertaan BPJS, dan merupakan **satu-satunya proksi sosial
ekonomi** yang tersedia di data ini:

| Kode | Arti |
|---|---|
| APBN / APBD | Iuran ditanggung pemerintah pusat / daerah |
| PPU PN / PPU BU | Pekerja penerima upah — penyelenggara negara / badan usaha |
| PBPU | Pekerja bukan penerima upah (mandiri) |
| BP PN / BP SWASTA | Bukan pekerja (termasuk pensiunan) |

Karena tidak punya urutan alami, segmen tidak bisa digambar sebagai garis tren. Bentuk yang
tepat adalah **peta panas**: seluruh penyakit sekaligus, sehingga polanya bisa dibandingkan.

In [ ]:

# ===================== PETA PANAS SEGMEN x PENYAKIT =====================
prev_sg, rr_sg, v_sg = {}, {}, {}
for nama in TARGET_INFO:
    sub, y = eligible(nama)
    p = y.groupby(sub["segmen"], observed=True).mean() * 100
    prev_sg[nama] = p
    rr_sg[nama] = p / (y.mean() * 100)          # risiko relatif terhadap rata-rata
    v_sg[nama] = cramers_v(sub["segmen"], y)

prev_sg = pd.DataFrame(prev_sg)
rr_sg = pd.DataFrame(rr_sg)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.heatmap(prev_sg, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[0],
            annot_kws={"fontsize": 11}, cbar_kws={"label": "% Berisiko"}, linewidths=0.5)
axes[0].set_title("Prevalensi mentah (%)\nbagus untuk beban layanan")
axes[0].set_ylabel("")

sns.heatmap(rr_sg, annot=True, fmt=".2f", cmap="RdYlGn_r", center=1.0, ax=axes[1],
            annot_kws={"fontsize": 11}, cbar_kws={"label": "risiko relatif"}, linewidths=0.5)
axes[1].set_title("Risiko relatif (1,00 = rata-rata penyakit itu)\nbagus untuk perbandingan")
axes[1].set_ylabel("")

fig.suptitle("Segmen Kepesertaan vs Keempat Risiko Penyakit", y=1.00)
plt.tight_layout()
plt.show()

print("Kekuatan asosiasi segmen dengan tiap penyakit (Cramer's V):")
for nama, v in v_sg.items():
    print(f"  {nama:<13} V = {v:.3f}  [{tafsir(v)}]")
print("\nRentang risiko relatif (tertinggi : terendah):")
for nama in TARGET_INFO:
    s = rr_sg[nama]
    print(f"  {nama:<13} {s.max()/s.min():.2f}x   "
          f"(tertinggi {s.idxmax()} {s.max():.2f}x, terendah {s.idxmin()} {s.min():.2f}x)")


Gradien antar `segmen` cukup jelas, terutama pada **TBC**. Karena `segmen` mencerminkan
status kepesertaan — dan secara tidak langsung status sosial ekonomi — pola ini konsisten
dengan literatur bahwa risiko TBC berkaitan erat dengan kondisi sosial ekonomi.

**Peringatan dalam membaca angka ini:** `segmen` juga berkorelasi dengan usia (misalnya
segmen "bukan pekerja" banyak berisi pensiunan). Jadi sebagian dari perbedaan ini adalah
efek usia yang menyamar sebagai efek sosial ekonomi. Tabel di Bagian I.7 memecah keduanya.


### Kenapa dua peta panas

**Peta kiri (prevalensi mentah)** menjawab *"berapa persen peserta di segmen ini yang
berisiko"*. Berguna untuk perencanaan beban layanan, tetapi tidak bisa dibandingkan antar
penyakit — kolom Hipertensi akan selalu terlihat merah karena prevalensi dasarnya memang 60%.

**Peta kanan (risiko relatif)** membagi tiap nilai dengan prevalensi penyakitnya sendiri,
sehingga **1,00 berarti sama dengan rata-rata** penyakit itu. Sekarang keempat kolom bisa
dibandingkan langsung: hijau = di bawah rata-rata, merah = di atas.

### Temuan

**TBC menunjukkan gradien paling lebar** — rentangnya sekitar 3,5 kali lipat antara segmen
tertinggi dan terendah, jauh melampaui tiga penyakit lain yang semuanya di bawah 1,7 kali.
Ini sejalan dengan III.6: segmen adalah variabel terkuat kedua untuk TBC dan bertahan setelah
usia dikendalikan.

**Hipertensi paling datar** (rentang 1,17x) — seluruh segmen praktis setara.

Perlu diingat dari III.2 bahwa segmen sangat terkait usia (η = 0.42), sehingga sebagian pola di
sini adalah efek usia. Untuk TBC hal itu sudah diuji secara khusus di III.6 dan hubungannya
bertahan; untuk penyakit lain gradiennya terlalu kecil untuk ditarik kesimpulan.

## III.8. Ringkasan Lintas Penyakit

In [ ]:

# ===================== MATRIKS ASOSIASI LENGKAP =====================
baris = []
for nama in TARGET_INFO:
    sub, y = eligible(nama)
    r = {"Target": nama}
    for c in NUM:
        r[c] = sub[c].corr(y)                 # point-biserial
    for c in CAT:
        r[c] = cramers_v(sub[c], y)           # phi / Cramer's V
    baris.append(r)

asosiasi = pd.DataFrame(baris).set_index("Target")

print("Numerik = point-biserial r (bertanda).  Kategorik = Cramer's V (0..1, tanpa arah).")
display(asosiasi.round(3))

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
sns.heatmap(asosiasi.abs(), annot=asosiasi.round(3), fmt="", cmap="YlOrRd",
            vmin=0, vmax=0.65, ax=ax, annot_kws={"fontsize": 12},
            cbar_kws={"label": "kekuatan asosiasi (nilai mutlak)"}, linewidths=0.5)
ax.set_title("Kekuatan Hubungan Tiap Variabel dengan Tiap Risiko Penyakit\n"
             "(warna = kekuatan, angka = nilai asli beserta tandanya)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:

# ===================== VARIABEL TERKUAT PER PENYAKIT =====================
ring = []
for nama in TARGET_INFO:
    s = asosiasi.loc[nama]
    urut = s.abs().sort_values(ascending=False)
    ring.append({
        "Target": nama,
        "Rentang usia": TARGET_INFO[nama][1],
        "Terkuat": urut.index[0], "Nilai": round(s[urut.index[0]], 3),
        "Tafsiran": tafsir(s[urut.index[0]]),
        "Kedua": urut.index[1], "Nilai 2": round(s[urut.index[1]], 3),
        "Ketiga": urut.index[2], "Nilai 3": round(s[urut.index[2]], 3),
    })

display(pd.DataFrame(ring).set_index("Target"))

---
# Kesimpulan

Empat blok di bawah ini adalah penutup masing-masing bagian, dikumpulkan di satu tempat
supaya seluruh temuan terbaca berurutan tanpa perlu melompat kembali ke tengah notebook.

## Profil populasi (Bagian I)


### Temuan utama

1. **Beban gizi ganda.** Underweight dan overweight/obese hadir bersamaan dalam populasi
   yang sama, sehingga intervensi gizi tidak bisa hanya menyasar satu arah.

2. **Gradien sosial ekonomi pada TBC.** Prevalensi risiko TBC berbeda nyata antar segmen
   kepesertaan, konsisten dengan kaitan TBC dan kondisi sosial ekonomi — meskipun sebagian
   efeknya bercampur dengan usia.

3. **Kanker Paru sangat timpang menurut jenis kelamin.** Perbedaan laki-laki vs perempuan
   jauh lebih besar daripada perbedaan antar kelompok usia, konsisten dengan kebiasaan
   merokok sebagai faktor dominan — meskipun status merokok sendiri tidak tersedia di data.

4. **Prevalensi risiko naik seiring usia** pada Diabetes, Hipertensi, dan TBC, tetapi
   relatif datar pada Kanker Paru — karena populasinya sudah terpotong pada usia 45+.

### Keterbatasan

- Semua angka **per catatan skrining**, bukan per orang; `id` teracak sehingga deduplikasi
  individu tidak mungkin.
- Angka prevalensi berlaku untuk **peserta yang ikut skrining**, bukan untuk seluruh
  penduduk Bantul — ada kemungkinan bias partisipasi.
- **Prevalensi "berisiko diabetes" tidak berdiri sendiri**: label tersebut praktis
  merupakan aturan `IMT > 25`, sehingga sebagian besar hanyalah pernyataan ulang dari
  prevalensi overweight.
- Faktor risiko penting tidak tersedia di data: riwayat keluarga, lingkar perut, status
  merokok, dan aktivitas fisik.

## Status gizi & klasifikasi BMI (Bagian II)

### Catatan penutup

1. **Dua standar dipakai karena satu ambang tidak berlaku untuk semua umur.** IMT anak
   berubah menurut usia dan jenis kelamin; memaksakan ambang dewasa akan salah
   mengklasifikasikan sebagian besar anak usia muda (lihat tabel rujukan CDC, Bagian II.1).

2. **Usia di bawah 2 tahun sengaja tidak diklasifikasi.** WHO merekomendasikan
   *Weight-for-Length* untuk kelompok ini, dan tabel rujukannya tidak tersedia dengan
   sumber yang dapat disitasi. Barisnya tetap dipertahankan karena relevan untuk TBC.

3. **Standar Indonesia (Kemenkes/IMT) tidak diimplementasikan** karena tidak ditemukan
   sumber yang dapat disitasi pada saat penyusunan. Bila rujukan resminya ditemukan,
   penambahannya cukup dilakukan di `bmi_standards.py` tanpa mengubah bagian ini.

4. **Kategori BMI bersifat redundan terhadap `IMT_recalculated`** — keduanya membawa
   informasi yang sama, hanya beda bentuk. Notebook modeling menguji hal ini secara
   eksplisit lewat ablasi fitur.

## Hubungan antar variabel (Bagian III)

## Jawaban ringkas atas pertanyaan awal

| Pertanyaan | Jawaban |
|---|---|
| **Apakah usia berhubungan dengan IMT?** | Ya bila semua usia digabung (*r* = +0.356), tetapi itu **efek pertumbuhan**. Pada dewasa saja praktis nol (*r* = −0.083) dan berbentuk lengkung |
| **Apakah usia berhubungan dengan risiko penyakit?** | Berbeda-beda. Paling jelas pada Diabetes (0.207) dan TBC (0.134); **hampir tidak ada** pada Hipertensi (0.054) dan Kanker Paru (−0.031) |
| **Apakah IMT berhubungan dengan risiko penyakit?** | Kuat pada Diabetes (0.616) — tetapi **melingkar**. Sedang pada Hipertensi (0.312) dengan bentuk mulus yang lebih dapat dipercaya. **Dapat diabaikan** pada Kanker Paru dan TBC |
| **Bagaimana tinggi dan berat berhubungan?** | Positif dan nyata, tetapi jauh lebih lemah daripada kesan awal: 0.797 semua usia, hanya **0.463** pada dewasa |
| **Apakah jenis kelamin atau segmen bermakna?** | **Jenis kelamin sangat bermakna untuk Kanker Paru saja** (V = 0.317). **Segmen paling bermakna untuk TBC** (V = 0.094) dan bertahan setelah usia dikendalikan |
| **Variabel mana yang terkuat untuk tiap penyakit?** | Diabetes → **IMT**; Hipertensi → **IMT**; Kanker Paru → **Jenis kelamin**; TBC → **Usia**, disusul **segmen** |

## Empat pola yang layak dibawa ke laporan

1. **Keempat penyakit punya profil yang berbeda.** Tidak ada satu variabel yang menjelaskan
   semuanya. Diabetes dan Hipertensi digerakkan komposisi tubuh, Kanker Paru oleh jenis
   kelamin, TBC oleh usia dan kondisi sosial ekonomi.

2. **Hubungan terkuat justru yang paling tidak informatif.** IMT ↔ Diabetes (0.616) adalah
   yang tertinggi di seluruh analisis, tetapi melingkar. Yang lebih layak dilaporkan justru
   IMT ↔ Hipertensi (0.312), karena bentuknya mulus dan tidak berasal dari aturan.

3. **TBC hampir tidak bisa dijelaskan oleh data yang tersedia.** Asosiasi tertinggi hanya
   0.134. Keterbatasannya ada pada **variabel yang tersedia**, bukan pada metode.

4. **Dari tiga hubungan yang tampak kuat, dua ternyata palsu.** Tinggi ↔ Kanker Paru
   sepenuhnya efek jenis kelamin (III.5); IMT ↔ TBC sebagian efek usia (III.6). Uji perancu bukan
   pelengkap — justru itu yang menentukan temuan mana yang boleh dilaporkan.

## Peringatan perbandingan

Kekuatan asosiasi **tidak sepenuhnya sebanding antar penyakit** karena populasinya berbeda.
Kanker Paru hanya mencakup usia 45+, sehingga variasi usianya kecil dan korelasi usianya
otomatis mengecil. TBC mencakup semua usia sehingga justru paling rawan perancu usia.

## Korelasi bukan sebab-akibat

Seluruh angka dalam bagian ini adalah **asosiasi**: dua hal cenderung muncul bersamaan.
Itu **tidak** membuktikan bahwa yang satu menyebabkan yang lain. Setidaknya ada empat
penjelasan lain untuk setiap hubungan di atas.

**1. Variabel perancu.** Sesuatu yang lain memengaruhi keduanya sekaligus. Sudah terbukti
tiga kali di bagian ini:

| Hubungan yang tampak | Setelah dikendalikan | Perancunya |
|---|---|---|
| Tinggi ↔ Berat = 0.797 | 0.463 | usia |
| Tinggi ↔ Kanker Paru = 0.203 | ~0.00 | jenis kelamin |
| IMT ↔ TBC = +0.066 | −0.030 s.d. +0.063 | usia |

**2. Arah sebab yang terbalik.** Korelasi tidak punya arah. Penyakit yang menyebabkan
penurunan berat badan akan terlihat sama saja dengan berat badan rendah yang menyebabkan
penyakit.

**3. Label yang bersifat aturan.** Kasus Diabetes adalah contoh ekstrem: labelnya *dihitung*
dari IMT. Korelasi 0.616 itu mengukur aturan penilaian kuesioner, bukan biologi.

**4. Bias partisipasi.** Data ini hanya memuat orang yang **ikut skrining**, bukan seluruh
penduduk Bantul. Orang yang merasa dirinya berisiko mungkin lebih terdorong ikut.

## Batasan tambahan

- **Semua angka berlaku per catatan skrining, bukan per orang.** Kolom `id` sengaja diacak
  untuk melindungi data asli BPJS, sehingga tidak mungkin memastikan satu orang hanya
  terhitung sekali.
- **Faktor risiko paling penting justru tidak tersedia**: status merokok, riwayat keluarga,
  lingkar perut, tekanan darah, dan aktivitas fisik. Jenis kelamin pada Kanker Paru hampir
  pasti hanyalah **penanda pengganti** untuk kebiasaan merokok.
- **Nilai-p sengaja tidak dilaporkan.** Pada n = 353.182, hubungan sekecil apa pun akan
  "signifikan"; yang relevan adalah besaran efeknya.

Dengan data yang ada, kesimpulan yang jujur berbentuk *"kelompok ini memiliki risiko lebih
tinggi"*, bukan *"karakteristik ini menyebabkan penyakit"*.

---
# Lampiran A — Metode & Cara Membaca

Bagian ini memuat dasar metodologis yang dipakai Bagian III. Isinya dipindah ke belakang
supaya alur analisis tidak terputus, bukan karena tidak penting — tabel ukuran di bawah
menentukan angka mana yang boleh dibandingkan dengan angka mana.

## Variabel yang dianalisis

| Variabel bebas | Tipe |
|---|---|
| `Jeniskelamin` | kategorik biner (L/P) |
| `Usia` | numerik |
| `Tinggibadan` | numerik |
| `Berat Badan` | numerik |
| `IMT_recalculated` | numerik |
| `segmen` | kategorik nominal (7 kategori) |

Empat target risiko: Diabetes Melitus, Hipertensi, Kanker Paru, TBC — semuanya **biner**.
## Metode: satu ukuran untuk tiap pasangan tipe

Tidak semua pasangan variabel bisa diukur dengan korelasi Pearson. Pearson mengasumsikan dua
variabel numerik dengan hubungan linear — memakainya pada `segmen` (yang tidak punya urutan)
akan menghasilkan angka yang tidak bermakna. Karena itu tiap pasangan memakai ukuran yang
sesuai:

| Pasangan | Ukuran | Rentang | Arti |
|---|---|---|---|
| numerik ↔ numerik | **Pearson** *r* | −1 … +1 | kekuatan hubungan **linear** |
| numerik ↔ numerik | **Spearman** ρ | −1 … +1 | kekuatan hubungan **monoton** (tahan outlier) |
| numerik ↔ target biner | **point-biserial** *r* | −1 … +1 | sama dengan Pearson memakai target 0/1 |
| biner ↔ biner | **phi** (φ) | 0 … 1 | kasus khusus Cramér's V untuk tabel 2×2 |
| nominal ↔ biner | **Cramér's V** | 0 … 1 | kekuatan asosiasi, tanpa arah |
| nominal ↔ numerik | **rasio korelasi** (η) | 0 … 1 | seberapa besar rata-rata berbeda antar kategori |

Ukuran tanpa arah (V dan η) tidak bisa negatif — ia hanya menyatakan **seberapa kuat**, bukan
ke arah mana.
## Panduan membaca besaran

| |nilai| | Tafsiran |
|---|---|
| < 0.10 | dapat diabaikan |
| 0.10 – 0.30 | lemah |
| 0.30 – 0.50 | sedang |
| > 0.50 | kuat |
## Kenapa bagian ini tidak memuat nilai-p

Dengan **353.182 baris**, hampir semua hubungan akan "signifikan secara statistik" — bahkan
korelasi sebesar 0.01 pun menghasilkan p < 0.001. Nilai-p pada ukuran sampel sebesar ini
hanya memberi tahu bahwa datanya banyak, bukan bahwa hubungannya penting.

Karena itu bagian ini melaporkan **besaran efek** (*effect size*), bukan signifikansi.

## Cara membaca grafik risiko

Sebelum masuk ke penyakit, satu kekeliruan perlu dijelaskan lebih dulu karena sangat mudah
terjadi dan membalik kesimpulan.

Cara yang terlihat masuk akal tetapi **keliru** adalah menyaring hanya orang yang berisiko,
lalu membuat histogram usianya. Tinggi tiap batang pada grafik semacam itu adalah:

```
jumlah berisiko  =  jumlah peserta seusia itu  x  risiko pada usia itu
```

Faktor pertama biasanya mendominasi, sehingga yang tergambar sebenarnya adalah **demografi
peserta skrining**, bukan risiko. Tiga panel berikut membedah persoalannya memakai TBC —
target yang rentang usianya paling lebar sehingga efeknya paling jelas.

In [ ]:

# ===================== JUMLAH vs PREVALENSI: KENAPA HITUNGAN MENYESATKAN =====================
sub_t, y_t = eligible("TBC")
bt = pd.cut(sub_t["Usia"], [0, 15, 25, 35, 45, 55, 65, 75, 120], right=False,
            labels=["<15", "15-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"])

ringkas_t = pd.DataFrame({
    "Diskrining": y_t.groupby(bt, observed=True).size(),
    "Berisiko": y_t.groupby(bt, observed=True).sum(),
})
ringkas_t["Prevalensi (%)"] = (ringkas_t["Berisiko"] / ringkas_t["Diskrining"] * 100).round(2)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
x = ringkas_t.index.astype(str)

axes[0].bar(x, ringkas_t["Diskrining"], color="#9aa6b2")
axes[0].set_title("1. Berapa banyak yang DISKRINING\n(penyebut)")
axes[0].set_ylabel("Jumlah peserta")
vz.thousands(axes[0])

axes[1].bar(x, ringkas_t["Berisiko"], color="#c0504d")
axes[1].set_title("2. KELIRU — jumlah yang berisiko\n(panel 1 x panel 3)")
axes[1].set_ylabel("Jumlah orang berisiko")
vz.thousands(axes[1])

axes[2].bar(x, ringkas_t["Prevalensi (%)"], color="#2d8746")
axes[2].set_title("3. BENAR — persen yang berisiko\n(ukuran risiko sesungguhnya)")
axes[2].set_ylabel("% Berisiko")

# Sorot dua kelompok yang ceritanya berbalik.
for ax, kol in [(axes[1], "Berisiko"), (axes[2], "Prevalensi (%)")]:
    for lbl in ["15-24", "75+"]:
        i = list(x).index(lbl)
        ax.patches[i].set_edgecolor("black")
        ax.patches[i].set_linewidth(2.5)
        v = ringkas_t.loc[lbl, kol]
        ax.annotate(f"{v:,.2f}" if kol.startswith("Prev") else f"{v:,}",
                    (i, v), ha="center", va="bottom", fontsize=12, weight="bold")

for ax in axes:
    ax.set_xlabel("Kelompok usia")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("TBC — kenapa histogram 'hanya yang berisiko' bukan ukuran risiko", y=1.00)
plt.tight_layout()
plt.show()

display(ringkas_t)


### Bandingkan dua kelompok yang ditandai kotak hitam

| Kelompok | Diskrining | Berisiko | Prevalensi |
|---|---|---|---|
| **15–24** | 45.197 | **1.115** | **2,47%** |
| **75+** | 12.520 | **1.004** | **8,02%** |

Pada panel 2 kedua batang ini **tingginya hampir sama** — seolah kedua kelompok sama-sama
bermasalah. Padahal kelompok 75+ **3,2 kali lebih berisiko**. Batangnya rendah semata-mata
karena lansia yang ikut skrining memang sedikit.

Kesalahan yang lebih berbahaya ada di ujung kanan panel 2: batangnya **memuncak di 55–64 lalu
menurun tajam**, sehingga pembaca akan menyimpulkan *"risiko TBC menurun setelah usia 65"*.
Panel 3 menunjukkan kebalikannya — risiko justru **tertinggi** di 65–74 (11,71%).

### Akibatnya untuk susunan grafik di Bagian III.3–III.6

Tiap penyakit digambarkan dengan **dua baris grafik yang saling melengkapi**:

| Baris | Bentuk | Menjawab |
|---|---|---|
| **Atas** | histogram **densitas**, dua kurva (Berisiko vs Tidak Berisiko) | *"di mana orangnya berada"* |
| **Bawah** | garis prevalensi per desil | *"seberapa besar risikonya"* |

Histogram di baris atas memakai **densitas**, bukan hitungan, supaya kedua kelompok bisa
dibandingkan meski jumlahnya sangat berbeda. Jarak antara kedua kurva menunjukkan seberapa
kuat variabel itu memisahkan risiko.

Kolomnya sejajar, jadi bisa dibaca ke bawah: *"banyak orang di rentang ini, dan risikonya
sekian"*.